# OMNI-X05 · Any video made ready on Colab's GPU, for the system to add with its vectors

The same code as the VIRAT archive (OMNI-708), imported from a clone of the branch, never copied
here: Algorithm 1 (`services/ingestion/app/pipeline.py`) with the product's settings, SigLIP 2
(`services/ai_workers/app/encoder.py`), InternVideo2 (`clip_encoder.py`), and the runners
(`scripts/encode_virat_archive.py`, `scripts/encode_uploads.py`).

1. **Setup:** clone the branch, install the pinned versions, mount Drive
2. **Load** the gate and both encoders, pinned (D-30, D-33), float32
3. **Pick videos:** every video in a Drive folder, and/or files uploaded to this session
4. **Encode:** every frame gated, both vectors for every admitted frame, one folder per video in
   `MyDrive/omnisight/ready/{fingerprint}/` (the SHA-256 of the file, D-47): the video file,
   `vectors.npz` and `info.json`
5. **Check:** each folder read back

Then on the laptop: `uv run python scripts/add_from_drive.py` lists the ready videos, and
`uv run python scripts/add_from_drive.py <numbers>` adds them to the system with both vectors
(no upload, no CPU encode).

**Run on a GPU:** Runtime → Change runtime type → GPU. Then Runtime → Run all. A rerun skips
videos already done.

In [ ]:
# 1 · Setup
BRANCH = "gpu-and-colab-upload"
REPO = "https://github.com/OmniSight-team/omni-sight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# The versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · Load the gate (Algorithm 1) and both encoders
from pathlib import Path
import encode_virat_archive as archive
import encode_uploads

REPO_DIR = Path("/content/omni-sight")
pipeline, GATE = archive.load_gate(REPO_DIR)
print("gate:", GATE)

from omnisight_common.settings import CommonSettings
from app.encoder import FrameTextEncoder, ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

pins = {name: CommonSettings.model_fields[name].default for name in
        ("frame_model_id", "frame_model_revision", "clip_model_id", "clip_model_revision")}
print("pins:", pins)

DRIVE = Path("/content/drive/MyDrive/omnisight")
MODELS = DRIVE / "models" / "pretrained"
OUT = DRIVE / "ready"

siglip_dir = ensure_checkpoint(pins["frame_model_id"], pins["frame_model_revision"], MODELS / "siglip2-so400m-patch14-384")
siglip = FrameTextEncoder(siglip_dir, device="cuda")
iv2_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
print(f"both encoders on {torch.cuda.get_device_name(0)}, {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

In [ ]:
# 3 · Pick videos: a folder on Drive (every video in it), and/or files uploaded here.
# Leave FOLDER as None to skip it; set UPLOAD = False to skip the upload box.
FOLDER = DRIVE / "uploads_to_encode"     # e.g. DRIVE / "data" / "ccd" / "videos"
UPLOAD = True
EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".mpg", ".mpeg", ".webm"}

videos = []
if FOLDER is not None and FOLDER.is_dir():
    videos += sorted(p for p in FOLDER.iterdir() if p.suffix.lower() in EXTENSIONS)
if UPLOAD:
    from google.colab import files
    LOCAL = Path("/content/uploaded"); LOCAL.mkdir(exist_ok=True)
    for name, data in files.upload().items():
        (LOCAL / name).write_bytes(data)
        videos.append(LOCAL / name)
print(f"{len(videos)} videos:", [v.name for v in videos])
assert videos, "no videos picked"

In [ ]:
# 4 · Encode: one folder per video, named by its fingerprint
summaries = []
for n, video in enumerate(videos, 1):
    print(f"[{n}/{len(videos)}]", end=" ")
    summaries.append(encode_uploads.encode_upload(
        video, OUT, pipeline, GATE, pins, siglip, clip_encoder, iv2, Path("/content/scratch")))

In [ ]:
# 5 · Check: each folder read back, its counts agreeing, unit vectors of the right size
import json
import numpy as np
for s in summaries:
    folder = OUT / s["fingerprint"]
    info = json.loads((folder / "info.json").read_text())
    assert (folder / info["file"]).is_file() and info["pins"] == pins and info["gate"] == GATE
    with np.load(folder / "vectors.npz") as data:
        n = len(data["frame_index"])
        assert data["frame_vectors"].shape == (n, 1152) and data["clip_vectors"].shape == (n, 512)
        for name in ("frame_vectors", "clip_vectors"):
            assert np.allclose(np.linalg.norm(data[name], axis=1), 1, atol=1e-3), name
    took = "" if s["skipped"] else f", {s['seconds']} s, {s['seconds'] / max(s['admitted'], 1):.3f} s per kept frame"
    print(f"{s['video']}: {s['fingerprint'][:12]}..., {n} frames kept{took}")
print("ready in", OUT)